In [ ]:
!pip install -r requirements.txt

4 Tâches à accomplir  
 
Ce qui suit est une série de questions liées à divers aspects du traitement des données.  
 
4.1 Data Loading:  Jeu de données tabulaires ou séries temporelles 
1.  Charger le jeu de donnees sélectionné (par exemple, dans un DataFrame Pandas en 
utilisant des fonctions appropriees telles que ”pd.read_csv()” dans le cas de jeux de 
donnees tabulaires/séries temporelles). 

In [ ]:
import pandas as pd

csv_path = './dataset/energydata_complete.csv'

df = pd.read_csv(csv_path)
print("Colonnes du dataset :", list(df.columns))
display(df.head())

# Signification des colonnes :
# - date : Date et heure de la mesure
# - Appliances : Consommation d’énergie (Wh) des appareils électroménagers
# - lights : Consommation d’énergie (Wh) de l’éclairage
# - T1 à T9 : Températures (°C) dans différentes pièces/zones (T1=salon, T2=cuisine, ...)
# - RH_1 à RH_9 : Humidité relative (%) dans les mêmes pièces/zones que T1 à T9
# - T_out : Température extérieure (°C)
# - Press_mm_hg : Pression atmosphérique (mm de mercure)
# - RH_out : Humidité relative extérieure (%)
# - Windspeed : Vitesse du vent (m/s)
# - Visibility : Visibilité (km)
# - Tdewpoint : Température du point de rosée (°C)
# - rv1, rv2 : Variables aléatoires (features artificielles, sans signification physique directe)
# Pour T1/RH_1 etc. :
#   - T1/RH_1 : Salon
#   - T2/RH_2 : Cuisine
#   - T3/RH_3 : Buanderie
#   - T4/RH_4 : Bureau
#   - T5/RH_5 : Salle de bain
#   - T6/RH_6 : Extérieur nord
#   - T7/RH_7 : Extérieur ouest
#   - T8/RH_8 : Garage
#   - T9/RH_9 : Chambre parentale

2.   Effectuer des tâches de pré-traitement des données telles que la gestion des 
valeurs manquantes, les conversions de types de données et le nettoyage des 
données.  

●  Vérifier les valeurs manquantes en utilisant ”df.isna()” et les traiter en les 
imputant ou en les supprimant. Assurez-vous d’expliquer les raisons derrière 
votre décision d’utiliser des techniques d'imputation ou de suppression pour 
traiter les données manquantes.

In [ ]:
# Vérification et traitement des valeurs manquantes
missing = df.isna().sum()
print('Valeurs manquantes par colonne :')
print(missing) # Pas de valeur manquante 

# Suppression des lignes très incomplètes (plus de 10% de NaN)
df = df.dropna(thresh=int(0.9*len(df.columns)))

# Imputation par la moyenne pour les colonnes numériques restantes
for col in df.select_dtypes(include='number').columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mean())
        print(f"Imputation par la moyenne pour : {col}")

# Justification :
# On supprime les lignes très incomplètes pour éviter de fausser l'analyse, puis on impute par la moyenne pour conserver le maximum d'information sans introduire de biais important.

In [ ]:
print(df.isna().sum())

●  Convertir les types de données au besoin en utilisant ”df.astype()”.

In [ ]:
print(df.dtypes) # Conversion possible : 'date' en datetime
df['date'] = pd.to_datetime(df['date'])

In [ ]:
print(df.dtypes)

●  Nettoyer les données manquantes en supprimant les doublons à l’aide de ”df.drop_duplicates()” et en corrigeant les valeurs incohérentes. Vous devez indiquer quelle technique a  été appliquée. Si vous choisissez l’imputation, précisez quelle méthode spécifique vous trouvez la plus appropriée pour votre jeu de données et pourquoi?

In [ ]:
# Suppression des doublons
avant = len(df)
df = df.drop_duplicates()
apres = len(df)
print(f"Nombre de doublons supprimés : {avant - apres}")

# Vérification de valeurs négatives sur les colonnes énergétiques
for col in df.select_dtypes(include='number').columns:
    if (df[col] < 0).any():
        print(f"Attention : valeurs négatives détectées dans {col}")

# Justification :
# On supprime les doublons pour éviter les biais et on signale les valeurs incohérentes pour garantir la qualité des données.

●  Créer de nouvelles fonctionnalités ou transformer celles existantes pour améliorer la qualité et la pertinence des données (si possible). 

In [ ]:
# Création de nouvelles fonctionnalités : différences de température intérieure/extérieure pour chaque pièce
T_pieces = [f'T{i}' for i in range(1, 10)]

for t_col in T_pieces:
    delta_col = f'delta_{t_col}_Tout'
    df[delta_col] = df[t_col] - df['T_out']
print(df[[col for col in df.columns if col.startswith('delta_T')]].head())

# Création d'une moyenne des températures intérieures
df['T_int_mean'] = df[T_pieces].mean(axis=1)
print(df[['T_int_mean'] + T_pieces].head())

# - delta_Tx_Tout : Différence entre la température de la pièce x (T1 à T9) et la température extérieure (T_out)
# - T_int_mean : Moyenne des températures intérieures (T1 à T9, hors T_out)

4.2 Analyse exploratoire des données (EDA) 
Profilage des données pour obtenir des informations sur leur distribution, leurs relations, leurs statistiques sommaires et les éventuels problèmes de qualité des données (données 
aberrantes).  
Dans cette partie, vous utiliserez Matplotlib, Seaborn pour créer une variété de graphiques, notamment :  
●  Des graphiques linéaires pour visualiser les tendances au fil du temps.

In [ ]:
# Graphiques linéaires pour visualiser plusieurs statistiques temporelles
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14,6))
plt.plot(df['date'], df['T1'], label='Température Salon (T1)')
plt.plot(df['date'], df['T_out'], label='Température extérieure (T_out)')
plt.xlabel('Date')
plt.ylabel('Température (°C)')
plt.title('Évolution des températures intérieure (T1) et extérieure (T_out)')
plt.legend()
plt.grid(True)
plt.show()

# Température moyenne intérieure
plt.figure(figsize=(14,6))
plt.plot(df['date'], df['T_int_mean'], label='Température intérieure moyenne')
plt.plot(df['date'], df['T_out'], label='Température extérieure (T_out)')
plt.xlabel('Date')
plt.ylabel('Température (°C)')
plt.title('Température intérieure moyenne vs extérieure')
plt.legend()
plt.grid(True)
plt.show()

# Consommation des appareils et des lumières
plt.figure(figsize=(14,6))
plt.plot(df['date'], df['Appliances'], label='Consommation appareils (Wh)', alpha=0.7)
plt.plot(df['date'], df['lights'], label='Consommation lumières (Wh)', alpha=0.7)
plt.xlabel('Date')
plt.ylabel('Consommation (Wh)')
plt.title('Consommation appareils et lumières dans le temps')
plt.legend()
plt.grid(True)
plt.show()

# Humidité relative intérieure moyenne vs extérieure
RH_cols = [col for col in df.columns if col.startswith('RH_') and col != 'RH_out']
df['RH_int_mean'] = df[RH_cols].mean(axis=1)
plt.figure(figsize=(14,6))
plt.plot(df['date'], df['RH_int_mean'], label='Humidité intérieure moyenne')
plt.plot(df['date'], df['RH_out'], label='Humidité extérieure')
plt.xlabel('Date')
plt.ylabel('Humidité relative (%)')
plt.title('Humidité intérieure moyenne vs extérieure')
plt.legend()
plt.grid(True)
plt.show()

●  Des graphiques de dispersion (scattering) pour identifier les relations entre les variables numériques. 

In [ ]:
# Graphiques de dispersion pour explorer plus de relations entre variables énergétiques
import matplotlib.pyplot as plt
import seaborn as sns

scatter_pairs = [
    ('T1', 'Appliances'),
    ('T_out', 'Appliances'),
    ('T1', 'T_out'),
    ('RH_1', 'Appliances'),
    ('T_int_mean', 'Appliances'),
    ('RH_int_mean', 'Appliances'),
    ('T1', 'lights'),
    ('T_out', 'lights'),
    ('T_int_mean', 'lights'),
    ('RH_1', 'lights'),
    ('RH_int_mean', 'lights'),
    ('T1', 'RH_1'),
    ('T_int_mean', 'RH_int_mean'),
    ('delta_T1_Tout', 'Appliances'),
    ('delta_T1_Tout', 'lights'),
]

for x, y in scatter_pairs:
    if x in df.columns and y in df.columns:
        plt.figure(figsize=(7,5))
        sns.scatterplot(data=df, x=x, y=y, alpha=0.6)
        plt.title(f'Scatter plot : {x} vs {y}')
        plt.xlabel(x)
        plt.ylabel(y)
        plt.grid(True)
        plt.show()

●  Des graphiques à barres pour comparer les données catégorielles.

In [ ]:
# Graphiques à barres pour explorer différentes statistiques catégorielles
import matplotlib.pyplot as plt
import seaborn as sns

# Consommation moyenne par heure
if 'hour' not in df.columns:
    df['hour'] = df['date'].dt.hour
hourly_mean = df.groupby('hour')['Appliances'].mean()
plt.figure(figsize=(10,5))
sns.barplot(x=hourly_mean.index, y=hourly_mean.values)
plt.title('Consommation moyenne des appareils par heure')
plt.xlabel('Heure')
plt.ylabel('Consommation moyenne (Wh)')
plt.show()

# Consommation totale par jour de la semaine
if 'weekday' not in df.columns:
    df['weekday'] = df['date'].dt.day_name()
weekday_sum = df.groupby('weekday')['Appliances'].sum().reindex([
    'Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.figure(figsize=(10,5))
sns.barplot(x=weekday_sum.index, y=weekday_sum.values)
plt.title('Consommation totale des appareils par jour de la semaine')
plt.xlabel('Jour de la semaine')
plt.ylabel('Consommation totale (Wh)')
plt.show()

# Moyenne de l'humidité intérieure par heure
if 'RH_int_mean' not in df.columns:
    RH_cols = [col for col in df.columns if col.startswith('RH_') and col != 'RH_out']
    if RH_cols:
        df['RH_int_mean'] = df[RH_cols].mean(axis=1)
if 'RH_int_mean' in df.columns:
    hourly_rh = df.groupby('hour')['RH_int_mean'].mean()
    plt.figure(figsize=(10,5))
    sns.barplot(x=hourly_rh.index, y=hourly_rh.values)
    plt.title("Humidité intérieure moyenne par heure")
    plt.xlabel('Heure')
    plt.ylabel('Humidité relative (%)')
    plt.show()

# Moyenne de la température intérieure par heure
if 'T_int_mean' in df.columns:
    hourly_temp = df.groupby('hour')['T_int_mean'].mean()
    plt.figure(figsize=(10,5))
    sns.barplot(x=hourly_temp.index, y=hourly_temp.values)
    plt.title("Température intérieure moyenne par heure")
    plt.xlabel('Heure')
    plt.ylabel('Température (°C)')
    plt.show()

●  Des cartes thermiques pour montrer les corrélations entre les variables.

In [ ]:
# Cartes thermiques fragmentées pour faciliter la lecture des corrélations
import matplotlib.pyplot as plt
import seaborn as sns

corr = df.select_dtypes(include='number').corr()

# Fonction pour afficher une heatmap sur une sous-matrice
def plot_corr_heatmap(sub_corr, title):
    plt.figure(figsize=(8,6))
    sns.heatmap(sub_corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# 1. Corrélations principales : consommation et températures
main_cols = ['Appliances', 'lights', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T_out']
plot_corr_heatmap(corr.loc[main_cols, main_cols], "Corrélations : Consommation & Températures")

# 2. Corrélations humidité
rh_cols = [col for col in corr.columns if col.startswith('RH_')]
if set(rh_cols).issubset(corr.columns):
    plot_corr_heatmap(corr.loc[rh_cols, rh_cols], "Corrélations : Humidité relative")

# 3. Corrélations météo
weather_cols = ['T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint']
weather_cols = [col for col in weather_cols if col in corr.columns]
if weather_cols:
    plot_corr_heatmap(corr.loc[weather_cols, weather_cols], "Corrélations : Variables météo")


●  Créer des visualisations interactives à l’aide de bibliothèques comme Plotly pour améliorer l'expérience utilisateur (bonus).

In [ ]:
# Visualisation interactive avec Plotly (bonus)
import plotly.express as px

fig = px.line(df, x='date', y=['T1', 'T_out'], title='Températures intérieure (T1) et extérieure (T_out) dans le temps')
fig.show()

Ensuite, vous devez : 
1.  Calculer des statistiques sommaires (à l'aide de fonctions comme ”df.describe()”) pour comprendre les tendances centrales de jeu de données et les distributions en utilisant des histogrammes, des traces de densité (KDE) et d’autres visualisations pour visualiser la distribution des données. 

In [ ]:
# Statistiques sommaires et histogrammes multiples
import matplotlib.pyplot as plt
import seaborn as sns

print(df.describe())

# Liste des colonnes numériques à visualiser (hors variables artificielles)
num_cols = [col for col in df.select_dtypes(include='number').columns if col not in ['rv1', 'rv2']]

# Affichage de plusieurs histogrammes sur une grille
n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f'Distribution de {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Effectif')

# Masquer les axes inutilisés
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

2.  Les valeurs aberrantes peuvent avoir un impact significatif sur l’analyse et les performances. Déterminer si les valeurs aberrantes sont des points de données valides ou des erreurs, et gérez-les en conséquence. Vous pouvez gérer les valeurs aberrantes en les visualisant à l’aide de boxplots et en décidant de les conserver ou 
de les supprimer. Les boxplot permettent de visualiser les quartiles, la médiane et les valeurs aberrantes dans les données.

In [ ]:
# Visualisation des valeurs aberrantes avec plusieurs boxplots
import matplotlib.pyplot as plt
import seaborn as sns

# Liste des colonnes numériques à visualiser (hors variables artificielles)
num_cols = [col for col in df.select_dtypes(include='number').columns if col not in ['rv1', 'rv2']]

# Affichage de plusieurs boxplots sur une grille
n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x=df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'Boxplot de {col}')
    axes[i].set_xlabel(col)

# Masquer les axes inutilisés
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

3.  Calculer les matrices de corrélation et les tracer pour identifier les relations entre les variables numériques. 

In [ ]:
# Corrélations : affichage sous forme de tableau et analyse automatique
import pandas as pd

corr = df.select_dtypes(include='number').corr()
print("Matrice de corrélation (extrait) :")
display(corr.round(2))

# Extraction des corrélations les plus fortes (hors diagonale)
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1]  # Exclure la diagonale
top_corr = corr_pairs.drop_duplicates().sort_values(ascending=False).head(10)
print("\nTop 10 des corrélations (en valeur absolue) :")
print(top_corr)

# Analyse automatique des résultats
print("\nAnalyse des corrélations :")
for (var1, var2), value in top_corr.items():
    sign = 'positive' if corr.loc[var1, var2] > 0 else 'négative'
    print(f"- {var1} et {var2} : corrélation {sign} de {corr.loc[var1, var2]:.2f}")

4.  Utiliser des techniques appropriées de détection d’anomalies statistiques telles que Z-score pour identifier les valeurs aberrantes/anomalies dans l’ensemble de données. Visualiser et analyser les anomalies détectées. Quelles conclusions pouvez-vous en tirer? (bonus). 

In [ ]:
# Détection d'anomalies avec Z-score sur toutes les variables numériques et analyse
from scipy.stats import zscore
import numpy as np

# Calcul du Z-score pour toutes les colonnes numériques
z_scores_df = df.select_dtypes(include='number').apply(zscore)

# Seuil classique pour détecter les outliers
threshold = 3

# Détection des anomalies pour chaque variable
anomalies_dict = {}
for col in z_scores_df.columns:
    anomalies = df[np.abs(z_scores_df[col]) > threshold][col]
    if not anomalies.empty:
        anomalies_dict[col] = anomalies

# Affichage des anomalies détectées
for col, values in anomalies_dict.items():
    print(f"\nAnomalies détectées pour {col} (Z-score > {threshold}) :")
    print(values)

# Synthèse et analyse
print("\nAnalyse des anomalies détectées :")
if anomalies_dict:
    for col, values in anomalies_dict.items():
        print(f"- {len(values)} anomalies détectées pour {col}.")
else:
    print("Aucune anomalie significative détectée selon le critère du Z-score.")

4.3 Manipulation des données 
1.  Appliquer le regroupement (grouping) et l'agrégation pour calculer des statistiques sommaires pour des catégories spécifiques. Vous pouvez utiliser la fonction groupby() et des fonctions d'agrégation telles que sum(), mean(), et count() pour créer des tables de synthèse. 

In [ ]:
# Regroupement et agrégation : statistiques sommaires par heure, jour et semaine
if 'date' in df.columns and 'Appliances' in df.columns:
    df = df.sort_values('date').copy()
    df['hour'] = df['date'].dt.hour
    df['weekday'] = pd.Categorical(
        df['date'].dt.day_name(),
        categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
        ordered=True
    )
    df['day'] = df['date'].dt.date

    hourly_stats = df.groupby('hour', observed=False)['Appliances'].agg(['mean', 'sum', 'count']).round(2)
    daily_stats = df.groupby('day')['Appliances'].agg(['mean', 'sum', 'count']).round(2)
    weekday_stats = df.groupby('weekday', observed=False)['Appliances'].agg(['mean', 'sum', 'count']).round(2)

    print('Statistiques par heure :')
    print(hourly_stats)

    peak_hour = hourly_stats['mean'].idxmax()
    peak_weekday = weekday_stats['mean'].idxmax()

    print('\nStatistiques par jour : graphique de la consommation moyenne par jour')
    plt.figure(figsize=(14, 5))
    plt.plot(daily_stats.index.astype(str), daily_stats['mean'], color='tab:blue', linewidth=1.5)
    plt.title('Consommation moyenne des appareils par jour')
    plt.xlabel('Jour')
    plt.ylabel('Consommation moyenne (Wh)')
    plt.xticks(rotation=45)
    plt.show()
    print('\nStatistiques par jour de la semaine :')
    print(weekday_stats)

    print(f'\nHeure la plus consommatrice en moyenne : {peak_hour} h')
    print(f'Jour le plus consommateur en moyenne : {peak_weekday}')
else:
    print("Colonnes nécessaires non trouvées pour le groupby.")


2.  Effectuer des opérations de filtrage ( à l’aide de l’indexation booléenne basée sur des conditions spécifiques) et des opérations de tri pour extraire des sous-ensembles de données (afin d’obtenir des informations).

In [ ]:
if 'hour' not in df.columns:
    df['hour'] = df['date'].dt.hour

if {'date', 'Appliances', 'lights', 'hour'}.issubset(df.columns):
    threshold = df['Appliances'].quantile(0.95)

    high_consumption = df[
        (df['Appliances'] >= threshold) &
        (df['lights'] > 0) &
        (df['hour'].between(17, 22))
    ][['date', 'hour', 'lights', 'Appliances']].sort_values(
        by=['Appliances', 'lights'],
        ascending=[False, False]
    )

    print(f"Seuil du 95e percentile pour Appliances : {threshold:.2f} Wh")
    print("\nExemples de pics de consommation en soirée avec éclairage allumé :")
    print(high_consumption.head(10))
    print(f"\nNombre total de lignes correspondant à ce filtre : {len(high_consumption)}")

else:
    print("Colonnes nécessaires non trouvées pour le filtrage/tri.")

3.  Appliquer des techniques d’analyse de séries temporelles pour découvrir des tendances temporelles dans le cas d’un jeu de données de séries temporelles. 
Utiliser des moyennes mobiles (rolling averages) ou d’autres fonctions de séries temporelles pour lisser le bruit dans les données (bonus). 

In [ ]:
if 'date' in df.columns and 'Appliances' in df.columns:
    df = df.sort_values('date').copy()

    # 24 pas = environ 4h si les mesures sont prises toutes les 10 minutes
    df['Appliances_MA24'] = df['Appliances'].rolling(window=24).mean()

    # 144 pas = environ 24h
    df['Appliances_MA144'] = df['Appliances'].rolling(window=144).mean()

    daily_consumption = df.set_index('date')['Appliances'].resample('D').mean()

    plt.figure(figsize=(14, 6))
    plt.plot(df['date'], df['Appliances'], label='Consommation réelle', alpha=0.35)
    plt.plot(df['date'], df['Appliances_MA24'], label='Moyenne mobile 24 pas (~4h)', linewidth=2)
    plt.plot(df['date'], df['Appliances_MA144'], label='Moyenne mobile 144 pas (~24h)', linewidth=2)
    plt.title("Consommation des appareils et moyennes mobiles")
    plt.xlabel("Date")
    plt.ylabel("Consommation (Wh)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Consommation moyenne journalière (5 premiers jours) :")
    print(daily_consumption.head())

    print(
        f"\nVariation entre le jour moyen le moins consommateur et le plus consommateur : "
        f"{daily_consumption.max() - daily_consumption.min():.2f} Wh"
    )

else:
    print("Colonnes nécessaires non trouvées pour l'analyse temporelle.")

4.4 Dérivation d’Informations  
1.  Assurez-vous d’avoir traité toutes les valeurs manquantes ou anomalies identifiées lors de l'examen du jeu de données.

In [ ]:
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

if 'Appliances_MA24' in df.columns:
    na_ma24 = df['Appliances_MA24'].isna().sum()
    print(f"\nValeurs manquantes induites par Appliances_MA24 : {na_ma24}")
    df['Appliances_MA24'] = df['Appliances_MA24'].bfill()

if 'Appliances_MA144' in df.columns:
    na_ma144 = df['Appliances_MA144'].isna().sum()
    print(f"Valeurs manquantes induites par Appliances_MA144 : {na_ma144}")
    df['Appliances_MA144'] = df['Appliances_MA144'].bfill()

print("\nVérification après traitement des moyennes mobiles :")
cols_to_check = [col for col in ['Appliances_MA24', 'Appliances_MA144'] if col in df.columns]
if cols_to_check:
    print(df[cols_to_check].isna().sum())

if 'z_scores_df' in globals():
    anomaly_summary = pd.DataFrame({
        'anomaly_count': (z_scores_df.abs() > 3).sum(),
        'anomaly_ratio_pct': ((z_scores_df.abs() > 3).mean() * 100).round(2)
    }).sort_values('anomaly_count', ascending=False)

    print("\nRésumé des anomalies détectées par Z-score :")
    print(anomaly_summary.head(10))

    print(
        "\nDécision : les anomalies détectées sont majoritairement conservées, "
        "car elles correspondent surtout à des pics plausibles de consommation "
        "ou d’éclairage, et non à des erreurs manifestes de saisie."
    )

for col in ['Appliances', 'T1', 'T_out']:
    if col in df.columns:
        print(f"{col} - min: {df[col].min()}, max: {df[col].max()}")

2.  Effectuer une analyse pour identifier les corrélations au sein d’un jeu de données sélectionné. Dans le cas d’un ensemble de données de séries temporelles, une analyse plus approfondie est nécessaire pour identifier la saisonnalité et les tendances au fil du temps. 

In [ ]:
import seaborn as sns

if 'Appliances' in df.columns:
    corr = df.corr(numeric_only=True)
    appliance_corr = corr['Appliances'].drop('Appliances').sort_values(ascending=False)

    print("Top 10 des corrélations positives avec Appliances :")
    print(appliance_corr.head(10))

    print("\nTop 10 des corrélations négatives avec Appliances :")
    print(appliance_corr.tail(10))

    # Corrélations les plus marquantes en valeur absolue
    strongest_corr = appliance_corr.reindex(appliance_corr.abs().sort_values(ascending=False).index).head(10)
    print("\nTop 10 des corrélations les plus fortes en valeur absolue :")
    print(strongest_corr)

    top_pos = appliance_corr.head(5)
    top_neg = appliance_corr.tail(5)
    corr_plot = pd.concat([top_pos, top_neg]).sort_values()

    plt.figure(figsize=(10, 6))
    plt.barh(corr_plot.index, corr_plot.values, color='steelblue')
    plt.title("Variables les plus corrélées à Appliances")
    plt.xlabel("Coefficient de corrélation")
    plt.ylabel("Variables")
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    if 'hour' not in df.columns:
        df['hour'] = df['date'].dt.hour

    if 'weekday' not in df.columns:
        df['weekday'] = pd.Categorical(
            df['date'].dt.day_name(),
            categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
            ordered=True
        )

    hourly_profile = df.groupby('hour')['Appliances'].mean().round(2)
    weekday_profile = df.groupby('weekday', observed=False)['Appliances'].mean().round(2)

    plt.figure(figsize=(12, 5))
    plt.plot(hourly_profile.index, hourly_profile.values, marker='o', color='tab:blue')
    plt.title("Saisonnalité intra-journalière de la consommation")
    plt.xlabel("Heure")
    plt.ylabel("Consommation moyenne (Wh)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.bar(weekday_profile.index.astype(str), weekday_profile.values, color='orange')
    plt.title("Consommation moyenne par jour de la semaine")
    plt.xlabel("Jour")
    plt.ylabel("Consommation moyenne (Wh)")
    plt.xticks(rotation=30)
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    seasonal_table = df.pivot_table(
        values='Appliances',
        index='hour',
        columns='weekday',
        aggfunc='mean'
    )

    plt.figure(figsize=(10, 6))
    sns.heatmap(seasonal_table, cmap='YlOrRd')
    plt.title("Heatmap de la consommation moyenne par heure et par jour")
    plt.xlabel("Jour de la semaine")
    plt.ylabel("Heure")
    plt.tight_layout()
    plt.show()

    peak_hour = hourly_profile.idxmax()
    low_hour = hourly_profile.idxmin()
    peak_day = weekday_profile.idxmax()
    low_day = weekday_profile.idxmin()

    print("\n--- Lecture des tendances ---")
    print(f"Pic moyen de consommation à {peak_hour} h")
    print(f"Creux moyen de consommation à {low_hour} h")
    print(f"Jour le plus consommateur en moyenne : {peak_day}")
    print(f"Jour le moins consommateur en moyenne : {low_day}")

else:
    print("Colonne 'Appliances' non trouvée pour l'analyse.")


3.  Interpréter les résultats pour tirer des conclusions significatives. 

In [ ]:
print("--- Interprétation des résultats ---")

if {'Appliances', 'hour', 'weekday'}.issubset(df.columns):
    corr = df.corr(numeric_only=True)['Appliances'].drop('Appliances').sort_values(ascending=False)
    hourly_profile = df.groupby('hour')['Appliances'].mean()
    weekday_profile = df.groupby('weekday', observed=False)['Appliances'].mean()

    top_positive = corr.head(3)
    top_negative = corr.tail(3)

    print(f"1. La consommation des appareils varie fortement au cours de la journée. Le pic moyen est observé à {hourly_profile.idxmax()} h, tandis que le niveau le plus faible apparaît à {hourly_profile.idxmin()} h.")

    print(f"2. À l’échelle hebdomadaire, le jour le plus consommateur en moyenne est {weekday_profile.idxmax()}, alors que le moins consommateur est {weekday_profile.idxmin()}.")

    print("3. Les moyennes mobiles montrent que la série brute est très fluctuante, mais qu’une tendance de fond plus régulière apparaît lorsque les données sont lissées.")

    print(f"4. Les variables les plus positivement corrélées à Appliances sont : {', '.join([f'{idx} ({val:.2f})' for idx, val in top_positive.items()])}.")

    print(f"5. Les variables les plus négativement corrélées à Appliances sont : {', '.join([f'{idx} ({val:.2f})' for idx, val in top_negative.items()])}.")

    print("6. Les anomalies détectées par Z-score correspondent surtout à des pics plausibles de consommation ou d’éclairage. Elles ont donc été conservées et interprétées comme des comportements réels.")

    print("7. Globalement, les analyses descriptives montrent que la consommation énergétique dépend à la fois du moment de la journée, du jour observé et de plusieurs variables du logement et de l’environnement. Cependant, les corrélations restent modérées, ce qui suggère qu’aucune variable prise isolément ne suffit à expliquer complètement la consommation.")

else:
    print("Certaines colonnes nécessaires à l’interprétation finale sont absentes.")